# Train the "Hey Evie / Hey EV / Evie / EV" wake word (Colab or Kaggle)

One small openWakeWord model that answers to all four phrasings, trained on Piper-synthesised speech (EV's own
`hfc_female` voice gets the biggest share) plus real LibriSpeech speech and ESC-50 noise as negatives.
Output: **`hey_evie.onnx`** (~400 KB). Drop it in `models/openwakeword/` and set `trigger.wakeword_model = "hey_evie"`.

**Setup**
* **Colab**: Runtime -> CPU is fine (no GPU needed). Run the cells top to bottom; the first one asks you to upload `ev_wakeword_kit.zip`.
* **Kaggle**: Settings -> **Internet: On**. Add `ev_wakeword_kit.zip` as a dataset (Add Data -> Upload; Kaggle unzips it, that's fine). Accelerator can stay *None*.

Takes roughly 1-2 h on CPU (synthesis and embedding dominate; training itself is a minute or two).

In [1]:
import glob, os, shutil, sys, zipfile

IN_KAGGLE = os.path.exists("/kaggle/working")
ROOT = "/kaggle/working/ev" if IN_KAGGLE else "/content/ev"
os.makedirs(ROOT, exist_ok=True)

def find_kit():
    # Kaggle unzips uploaded datasets into a folder, Colab keeps the zip
    folder = glob.glob("/kaggle/input/**/scripts/wakeword/synth.py", recursive=True)
    if folder:
        shutil.copytree(folder[0].rsplit("/scripts/", 1)[0], ROOT, dirs_exist_ok=True)
        return True
    for z in glob.glob("/kaggle/input/**/ev_wakeword_kit.zip", recursive=True) + glob.glob("/content/ev_wakeword_kit.zip"):
        zipfile.ZipFile(z).extractall(ROOT)
        return True
    return False

if not find_kit() and not IN_KAGGLE:
    from google.colab import files
    files.upload()                      # pick ev_wakeword_kit.zip
    find_kit()
assert os.path.exists(f"{ROOT}/scripts/wakeword/synth.py"), "kit not found: upload ev_wakeword_kit.zip first"
os.chdir(ROOT)
print("working in", os.getcwd(), "| cpus:", os.cpu_count())

working in /content/ev | cpus: 2


In [2]:
# Same versions as the local EV install. openwakeword goes in with --no-deps so it doesn't try to pull tflite on Linux.
!pip install -q piper-tts==1.8.0 kokoro-onnx==0.6.1 soundfile scipy onnx onnxruntime scikit-learn requests tqdm
!pip install -q --no-deps openwakeword==0.6.0
import torch; print("torch", torch.__version__)

torch 2.11.0+cpu


In [3]:
# Models: 6 Piper voices (hfc_female = EV's main voice), openWakeWord's shared feature models, Kokoro (evaluation only).
import subprocess
voices = ["en_US-hfc_female-medium", "en_US-amy-medium", "en_US-lessac-medium", "en_US-ryan-medium",
          "en_GB-alan-medium", "en_US-libritts_r-medium"]
for i, v in enumerate(voices):
    cmd = [sys.executable, "scripts/download_models.py", "--voice", v] + (["--wakeword", "hey_jarvis"] if i == 0 else [])
    subprocess.run(cmd, check=True)
os.makedirs("models/kokoro", exist_ok=True)
K = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0"
!wget -q -nc -P models/kokoro {K}/kokoro-v1.0.onnx {K}/voices-v1.0.bin
!ls -la models/piper models/openwakeword models/kokoro | grep -v json

models/kokoro:
total 345468
drwxr-xr-x 2 root root      4096 Oct 29 16:11 .
drwxr-xr-x 5 root root      4096 Oct 29 16:11 ..
-rw-r--r-- 1 root root 325532387 Jan 28  2025 kokoro-v1.0.onnx
-rw-r--r-- 1 root root  28214398 Feb  9  2025 voices-v1.0.bin

models/openwakeword:
total 3612
drwxr-xr-x 2 root root    4096 Oct 29 16:11 .
drwxr-xr-x 5 root root    4096 Oct 29 16:11 ..
-rw-r--r-- 1 root root 1326578 Oct 29 16:11 embedding_model.onnx
-rw-r--r-- 1 root root 1271370 Oct 29 16:11 hey_jarvis_v0.1.onnx
-rw-r--r-- 1 root root 1087958 Oct 29 16:11 melspectrogram.onnx

models/piper:
total 385428
drwxr-xr-x 2 root root     4096 Oct 29 16:11 .
drwxr-xr-x 5 root root     4096 Oct 29 16:11 ..
-rw-r--r-- 1 root root 63201294 Oct 29 16:11 en_GB-alan-medium.onnx
-rw-r--r-- 1 root root 63201294 Oct 29 16:11 en_US-amy-medium.onnx
-rw-r--r-- 1 root root 63201294 Oct 29 16:10 en_US-hfc_female-medium.onnx
-rw-r--r-- 1 root root 63201294 Oct 29 16:11 en_US-lessac-medium.onnx
-rw-r--r-- 1 root root 78580

In [4]:
# Public negatives: LibriSpeech dev-clean (train) + test-clean (held out) + ESC-50 noise. ~1.3 GB, a few minutes.
!python scripts/wakeword/fetch_data.py

librispeech-dev: downloading https://www.openslr.org/resources/12/dev-clean.tar.gz
librispeech-dev: extracting
librispeech-test: downloading https://www.openslr.org/resources/12/test-clean.tar.gz
librispeech-test: extracting
esc50: downloading https://github.com/karoldvl/ESC-50/archive/master.zip
esc50: extracting
done


In [5]:
# Piper synthesis: positives, near-miss words ("every", "Eddie", "TV", "hey Siri"...), generic speech. Resumable.
# Use fewer clips (e.g. --pos 4000 --adv 3000 --gen 3000) for a faster first pass.
!python scripts/wakeword/synth.py --pos 8000 --adv 6000 --gen 6000

20000 clips with 1 workers
  500/20000
  1000/20000
  1500/20000
  2000/20000
  2500/20000
  3000/20000
  3500/20000
  4000/20000
  4500/20000
  5000/20000
  5500/20000
  6000/20000
  6500/20000
  7000/20000
  7500/20000
  8000/20000
  8500/20000
  9000/20000
  9500/20000
  10000/20000
  10500/20000
  11000/20000
  11500/20000
  12000/20000
  12500/20000
  13000/20000
  13500/20000
  14000/20000
  14500/20000
  15000/20000
  15500/20000
  16000/20000
  16500/20000
  17000/20000
  17500/20000
  18000/20000
  18500/20000
  19000/20000
  19500/20000
  20000/20000
done: 20000 clips in /content/ev/data/wakeword/synth


In [6]:
# Augment (noise, reverb, EQ, pitch/speed) and compute the 16x96 openWakeWord embeddings.
# If data/wakeword/recordings/{pos,neg} was in the kit, your own voice is mixed in here (very worthwhile).
!python scripts/wakeword/features.py --reps 3

train: 28188 base clips
  train: 216/28188
  train: 448/28188
  train: 681/28188
  train: 908/28188
  train: 1124/28188
  train: 1350/28188
  train: 1576/28188
  train: 1790/28188
  train: 2008/28188
  train: 2226/28188
  train: 2445/28188
  train: 2679/28188
  train: 2902/28188
  train: 3139/28188
  train: 3375/28188
  train: 3595/28188
  train: 3826/28188
  train: 4052/28188
  train: 4281/28188
  train: 4503/28188
  train: 4721/28188
  train: 4943/28188
  train: 5163/28188
  train: 5391/28188
  train: 5610/28188
  train: 5834/28188
  train: 6061/28188
  train: 6277/28188
  train: 6499/28188
  train: 6731/28188
  train: 6952/28188
  train: 7170/28188
  train: 7402/28188
  train: 7621/28188
  train: 7847/28188
  train: 8078/28188
  train: 8302/28188
  train: 8535/28188
  train: 8757/28188
  train: 8987/28188
  train: 9207/28188
  train: 9423/28188
  train: 9632/28188
  train: 9861/28188
  train: 10099/28188
  train: 10321/28188
  train: 10543/28188
  train: 10775/28188
  train: 11006/2

In [7]:
!python scripts/wakeword/train.py --epochs 60

train (64188, 16, 96) (21600 pos)   val (2569, 16, 96) (800 pos)
102,849 parameters
ep   1  loss 0.1998  recall 0.924  fp 0.0458
ep   2  loss 0.1770  recall 0.906  fp 0.0220
ep   3  loss 0.1343  recall 0.976  fp 0.0701
ep   4  loss 0.1427  recall 0.939  fp 0.0288
ep   5  loss 0.0698  recall 0.950  fp 0.0288
ep   6  loss 0.1601  recall 0.943  fp 0.0249
ep   7  loss 0.0852  recall 0.936  fp 0.0220
ep   8  loss 0.1293  recall 0.935  fp 0.0187
ep   9  loss 0.0994  recall 0.971  fp 0.0334
ep  10  loss 0.0817  recall 0.939  fp 0.0153
ep  11  loss 0.0998  recall 0.956  fp 0.0277
ep  12  loss 0.1568  recall 0.932  fp 0.0153
ep  13  loss 0.0985  recall 0.930  fp 0.0198
ep  14  loss 0.1145  recall 0.941  fp 0.0187
ep  15  loss 0.1245  recall 0.931  fp 0.0181
ep  16  loss 0.0633  recall 0.954  fp 0.0300
ep  17  loss 0.0802  recall 0.891  fp 0.0119
ep  18  loss 0.1156  recall 0.949  fp 0.0243
ep  19  loss 0.0646  recall 0.964  fp 0.0266
ep  20  loss 0.0690  recall 0.916  fp 0.0175
ep  21  loss 0.1

In [ ]:
# Streamed through openwakeword.Model like EV does: recall on UNSEEN Kokoro voices, false triggers/hour on
# LibriSpeech test-clean + unseen noise. Pick a threshold from this table for trigger.wakeword_threshold.
!python scripts/wakeword/evaluate.py --hours 2

model: hey_evie   thresholds: [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

recall on held-out Kokoro voices (fraction of clips whose peak score reaches the threshold)
  phrase            0.3    0.4    0.5    0.6    0.7    0.8    0.9
  E V              1.00   1.00   1.00   1.00   1.00   1.00   1.00
  ee vee           1.00   1.00   1.00   1.00   1.00   1.00   1.00
  eevee            0.88   0.86   0.86   0.86   0.86   0.83   0.81
  evie             0.93   0.90   0.90   0.90   0.88   0.88   0.86
  evvy             1.00   1.00   1.00   1.00   1.00   0.98   0.95
  hey E V          1.00   1.00   1.00   1.00   1.00   1.00   1.00
  hey ee vee       1.00   1.00   1.00   1.00   1.00   1.00   1.00
  hey eevee        1.00   1.00   1.00   1.00   1.00   1.00   1.00
  hey evie         0.98   0.98   0.98   0.98   0.95   0.95   0.95
  hey evvy         1.00   1.00   1.00   1.00   1.00   1.00   1.00
  ALL              0.98   0.97   0.97   0.97   0.97   0.96   0.96


In [ ]:
# Grab the model. Colab downloads it; on Kaggle it appears under the notebook's Output tab.
src = "models/openwakeword/hey_evie.onnx"
if IN_KAGGLE:
    shutil.copy(src, "/kaggle/working/hey_evie.onnx")
    print("saved /kaggle/working/hey_evie.onnx (Output tab, after 'Save Version')")
else:
    from google.colab import files
    files.download(src)

**Next (on your PC):** copy `hey_evie.onnx` to `models/openwakeword/`, set in `config/ev.toml`
`wakeword_model = "hey_evie"` and a `wakeword_threshold` from the table above, then `python -m ev --debug`.